In [1]:
from src.document_loader import load_document

Documents = load_document("data/Employee_Benefits.docx")

print(Documents)

[Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data\\Employee_Benefits.docx'}, page_content='Employee Benefits\n\nEmployees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage continuous learning and career advancement

In [2]:
from src.chunking import split_documents

chunks = split_documents(Documents, chunk_size=400, chunk_overlap=50)

print(chunks)

[Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 0}, page_content='Employee Benefits'), Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 1}, page_content='Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,'), Document(metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data

In [3]:
import json
from src.retrieval.embedding import get_embedding_model, VectorStoreManager
from langchain_core.documents import Document

with open("config/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print(config["llm_provider"])
print(config["embedding_provider"])
print(config["embedding_model"])

embedding_model = get_embedding_model("openai",config["embedding_model"])
print(embedding_model)

from langchain_community.vectorstores import FAISS, Chroma

manager = VectorStoreManager(config=config)
len(chunks)
vector_store = manager.create_index(chunks)

openai
openai
text-embedding-3-small
Loading OpenAI embedding model...
client=<openai.resources.embeddings.Embeddings object at 0x0000021B6CE63090> async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x0000021B6CF90A50> model='text-embedding-3-small' dimensions=None deployment='text-embedding-ada-002' openai_api_version=None openai_api_base=None openai_api_type=None openai_proxy=None embedding_ctx_length=8191 openai_api_key=SecretStr('**********') openai_organization=None allowed_special=None disallowed_special=None chunk_size=1000 max_retries=2 request_timeout=None headers=None tiktoken_enabled=True tiktoken_model_name=None show_progress_bar=False model_kwargs={} skip_empty=False default_headers=None default_query=None retry_min_seconds=4 retry_max_seconds=20 http_client=None http_async_client=None check_embedding_ctx_length=True
Loading OpenAI embedding model...


In [4]:
results = vector_store.similarity_search(
    "What health insurance coverage is available?",
    k=3
)

for r in results:
    print(r.page_content)
    print(r.metadata)

Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,
{'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 1}
medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage
{'source': 'd:\\AI and ML\\IIT pa

In [5]:
results = vector_store.similarity_search_with_score(
    "health insurance",
    k=3
)

for doc, score in results:
    print(f"Score: {score}")
    print(doc.page_content)

Score: 0.9437819719314575
medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage
Score: 0.9767372608184814
Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,
Score: 1.080682396888733
Employee Benefits


In [6]:
from src.retrieval.bm25_search import BM25Retriever

bm25retriver = BM25Retriever(chunks)

query = "What health insurance coverage is available?"

ranked_docs = bm25retriver.search(
    query=query,
    k=3
)

for doc, score in ranked_docs:
    print("=" * 50)
    print(f"Score: {score:.4f}")
    print(doc.page_content)
    print(doc.metadata)

Score: 1.0850
Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,
{'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 1}
Score: 1.0331
medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage
{'sou

In [7]:
from src.retrieval.hybrid_retriever import HybridRetriever

hybrid = HybridRetriever(
    vector_manager=manager,
    bm25_retriever=bm25retriver
)

results = hybrid.retrieve(
    "What health insurance coverage is available?"
)

for i, doc in enumerate(results, start=1):

    print(f"\nRank {i}")

    print(doc.page_content)

    print(doc.metadata)


Rank 1
Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,
{'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 1}

Rank 2
medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. To encourage
{'source': 'd:\\A

In [8]:
from src.retrieval.llm_based_reranker import LLMReranker

query = "What health insurance coverage is available?"

llmreranker = LLMReranker()
top_reranked_docs = llmreranker.rerank_with_scores(query,results)

for docs,score in top_reranked_docs:
    print(score)
    print(docs)

8.0
page_content='Employees are provided with a comprehensive benefits package designed to support their health, financial security, professional development, and overall well-being. The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations,' metadata={'source': 'd:\\AI and ML\\IIT patna\\GenAI\\Assignment\\Enterprise_Knowledge_Assistant_with_Advanced-RAG\\enterprise-agentic-rag\\data\\Employee_Benefits.docx', 'chunk_index': 1}
5.0
page_content='medical treatments, specialist consultations, emergency care, and other eligible healthcare services. In addition to health protection, employees are covered by a life insurance policy that provides financial security for their loved ones, with coverage valued at three times the employee’s annual salary, ensuring meaningful support in the event of an unforeseen circumstance. 

In [9]:
from src.retrieval.hallucination_guard import HallucinationGuard

guard = HallucinationGuard()

answer = """
Health insurance coverage is up to $150,000.
Life insurance coverage is equal to three times annual salary.
"""

report = guard.evaluate(
    summary=answer,
    docs = Documents)

print(report)



[('Health insurance coverage is up to $150,000.', 0.302, '✅ Supported', False), ('Life insurance coverage is equal to three times annual salary.', 0.251, '✅ Supported', False)]


In [10]:
from src.memory.conversation_memory import ConversationMemory

memory = ConversationMemory()

memory.add_interaction(
    "What is the leave policy?",
    "Employees receive 24 annual leave days."
)

memory.add_interaction(
    "Can I carry them forward?",
    "Up to 10 days can be carried forward."
)

print(
    memory.get_context()
)

User: What is the leave policy?
Assistant: Employees receive 24 annual leave days.
User: Can I carry them forward?
Assistant: Up to 10 days can be carried forward.


In [11]:
from src.pipeline.rag_pipeline import RAGPipeline

pipeline = RAGPipeline(
    retriever=hybrid,
    reranker=llmreranker,
    memory=memory,
    hallucination_guard=guard
)

response = pipeline.ask(
    "What health insurance coverage is available?"
)

print(response["answer"])



The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations, emergency care, and other eligible healthcare services. [d:\AI and ML\IIT patna\GenAI\Assignment\Enterprise_Knowledge_Assistant_with_Advanced-RAG\enterprise-agentic-rag\data\Employee_Benefits.docx]
The health insurance plan offers extensive medical coverage with benefits of up to $150,000, helping employees and their families manage healthcare expenses related to hospitalization, medical treatments, specialist consultations, emergency care, and other eligible healthcare services.
